# TB classifier: remote experiments
Runtime → Change runtime type → **T4 GPU**, then Runtime → **Run all**. The last cell downloads `runs_<names>.zip`; unzip it in the project folder on the Mac.

In [ ]:
!nvidia-smi -L
!pip -q install timm grad-cam kagglehub torchxrayvision opencv-python-headless

In [ ]:
%%writefile prepare.py
"""Download TBX11K, resize to 256px grayscale, write data/{train,val,test,external}.npz + data/boxes.json."""
import json
from collections import defaultdict
from pathlib import Path

import kagglehub
import numpy as np
from PIL import Image
from sklearn.model_selection import train_test_split

SIZE = 256
OUT = Path("data")
ROOT = Path(kagglehub.dataset_download("usmanshams/tbx-11")) / "TBX11K"


def tbx_list(name):
    rels = (ROOT / "lists" / name).read_text().split()
    return [(ROOT / "imgs" / r, int(r.startswith("tb/")), "tbx11k") for r in rels]


def extra_label(p):
    if p.parent.parent.name == "mc+shenzhen":
        return int(p.stem.endswith("_1"))  # MCUCXR_0023_0 / CHNCXR_0636_1
    return int(p.stem.startswith("p"))  # DA/DB: n*/nx* normal, p*/px* TB


val_all = tbx_list("TBX11K_val.txt")
val, test = train_test_split(val_all, test_size=0.5, stratify=[y for _, y, _ in val_all], random_state=0)
extra = [(p, extra_label(p), p.parent.parent.name) for p in sorted((ROOT / "imgs/extra").glob("*/*/*.*"))]
splits = {"train": tbx_list("TBX11K_train.txt"), "val": val, "test": test, "external": extra}


def thumbs(x):
    t = x.reshape(len(x), 32, SIZE // 32, 32, SIZE // 32).mean((2, 4)).reshape(len(x), -1)
    return t.astype(np.float32)


def min_dist(a, b):
    """Per-pixel RMS distance from each row of a to its nearest row in b."""
    d2 = (a**2).sum(1)[:, None] + (b**2).sum(1)[None] - 2 * a @ b.T
    return np.sqrt(np.maximum(d2.min(1), 0) / a.shape[1])


OUT.mkdir(exist_ok=True)
# Official TBX11K splits contain (near-)identical copies across train/val; drop later-split copies to avoid leakage.
DUP_RMS = 8.0  # 0-255 scale; observed: dup pairs <= 3.6, nearest distinct pair ~14
seen = np.zeros((0, 32 * 32), np.float32)
for name, items in splits.items():
    x = np.stack([np.asarray(Image.open(p).convert("L").resize((SIZE, SIZE), Image.BILINEAR)) for p, _, _ in items])
    t = thumbs(x)
    keep = min_dist(t, seen) > DUP_RMS if len(seen) else np.ones(len(x), bool)
    seen = np.concatenate([seen, t[keep]])
    items = [it for it, k in zip(items, keep) if k]
    y = np.array([y for _, y, _ in items])
    np.savez(OUT / f"{name}.npz", x=x[keep], y=y,
             path=[str(p.relative_to(ROOT / "imgs")) for p, _, _ in items],
             source=[s for _, _, s in items])
    print(f"{name:9s} n={len(y):5d}  TB={y.sum():4d}  dropped_dups={(~keep).sum()}")

# COCO bboxes (512px coords, xywh) for TB images, keyed by relative path
boxes = defaultdict(list)
for f in ["TBX11K_train.json", "TBX11K_val.json"]:
    j = json.loads((ROOT / "annotations/json" / f).read_text())
    names = {im["id"]: im["file_name"] for im in j["images"]}
    for a in j["annotations"]:
        boxes[names[a["image_id"]]].append(a["bbox"])
(OUT / "boxes.json").write_text(json.dumps(boxes))
print(f"bbox images: {len(boxes)}")


In [ ]:
%%writefile train.py
"""Fine-tune a pretrained CNN for TB vs non-TB on data/*.npz (from prepare.py). Tuned for a 16GB MacBook Air (MPS); also runs on CUDA (Colab)."""
import argparse
import json
import time
from pathlib import Path

import cv2
import numpy as np
import timm
import torch
import torch.nn.functional as F
import torchxrayvision as xrv
from sklearn.metrics import confusion_matrix, roc_auc_score, roc_curve
from torch import nn
from torchvision.transforms import v2
from tqdm import tqdm

DEVICE = "cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu"


def load_split(name, clahe=False):
    z = np.load(f"data/{name}.npz")
    x = z["x"]
    if clahe:  # local contrast equalization: evens out scanner/processing differences between hospitals
        c = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
        x = np.stack([c.apply(img) for img in x])
    return torch.from_numpy(x), torch.from_numpy(z["y"]), z["path"], z["source"]


class Scale(nn.Module):
    """Maps [0,1] pixels to whatever range the backbone was pretrained on."""

    def __init__(self, mul, add):
        super().__init__()
        self.mul, self.add = mul, add

    def forward(self, x):
        return x * self.mul + self.add


def build_model(name, pretrained=True):
    """Returns Sequential(Scale, net); checkpoints store net only, so model[1] is the backbone."""
    if name == "xrv-densenet":  # DenseNet121 pretrained on ~8 public chest X-ray datasets from many hospitals
        net = xrv.models.DenseNet(weights="densenet121-res224-all") if pretrained else xrv.models.DenseNet(num_classes=1)
        net.op_threshs = None
        net.classifier = nn.Linear(net.classifier.in_features, 1)
        net.input_resolution = 224  # xrv resizes internally to its native resolution
        return nn.Sequential(Scale(2048, -1024), net)  # xrv expects [-1024, 1024]
    mean, std = 0.449, 0.226  # ImageNet RGB stats averaged, since timm folds RGB weights into 1 channel
    net = timm.create_model(name, pretrained=pretrained, num_classes=1, in_chans=1)
    return nn.Sequential(Scale(1 / std, -mean / std), net)


def to_input(x_uint8):
    return (x_uint8.float() / 255).unsqueeze(1)


# No horizontal flip: heart/aorta side is anatomical signal, flipping makes fake anatomy.
AUGMENT = v2.Compose([
    v2.RandomAffine(degrees=7, translate=(0.05, 0.05), scale=(0.9, 1.1)),
    v2.ColorJitter(brightness=0.2, contrast=0.2),
])
# Simulates other scanners: exposure curve (gamma), contrast, softness/sharpness of the detector.
STRONG_AUGMENT = v2.Compose([
    v2.RandomAffine(degrees=10, translate=(0.08, 0.08), scale=(0.85, 1.15)),
    v2.Lambda(lambda x: v2.functional.adjust_gamma(x, float(np.random.uniform(0.7, 1.5)))),
    v2.ColorJitter(brightness=0.4, contrast=0.4),
    v2.RandomApply([v2.GaussianBlur(5, sigma=(0.1, 1.5))], p=0.3),
    v2.RandomAdjustSharpness(2, p=0.3),
])


@torch.no_grad()
def predict(model, x, bs=128):
    model.eval()
    return torch.cat([torch.sigmoid(model(to_input(x[i:i + bs]).to(DEVICE))).squeeze(1).cpu()
                      for i in range(0, len(x), bs)]).numpy()


def metrics(y, p, thr):
    tn, fp, fn, tp = confusion_matrix(y, p >= thr, labels=[0, 1]).ravel()
    return {"auc": round(roc_auc_score(y, p), 4), "sensitivity": round(tp / (tp + fn), 4),
            "specificity": round(tn / (tn + fp), 4), "tp": int(tp), "fn": int(fn), "fp": int(fp), "tn": int(tn)}


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--name", default="baseline", help="experiment folder under runs/")
    ap.add_argument("--model", default="resnet18", help="any timm model, or xrv-densenet")
    ap.add_argument("--clahe", action="store_true")
    ap.add_argument("--strong-aug", action="store_true")
    ap.add_argument("--epochs", type=int, default=10)
    ap.add_argument("--batch", type=int, default=32)  # 32 @256px fits 16GB unified memory comfortably
    args = ap.parse_args()
    cfg = vars(args)
    runs = Path("runs") / args.name
    runs.mkdir(parents=True, exist_ok=True)
    print(cfg)

    torch.manual_seed(0)
    np.random.seed(0)
    xtr, ytr, _, _ = load_split("train", args.clahe)
    xva, yva, _, _ = load_split("val", args.clahe)
    augment = STRONG_AUGMENT if args.strong_aug else AUGMENT
    model = build_model(args.model).to(DEVICE)
    opt = torch.optim.AdamW(model.parameters(), lr=3e-4, weight_decay=1e-4)
    steps = args.epochs * (len(xtr) // args.batch)
    sched = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=3e-4, total_steps=steps, pct_start=0.1)
    pos_weight = ((ytr == 0).sum() / (ytr == 1).sum()).float().to(DEVICE)  # ~10x: TB is rare
    best = -1

    for ep in range(args.epochs):
        model.train()
        t0, perm, total = time.time(), torch.randperm(len(xtr)), 0.0
        bar = tqdm(range(0, len(perm) - args.batch + 1, args.batch), desc=f"ep {ep + 1}/{args.epochs}", leave=False)
        for i in bar:  # drop last partial batch
            idx = perm[i:i + args.batch]
            xb = torch.stack([augment(img[None]) for img in xtr[idx]]).squeeze(1)
            xb, yb = to_input(xb).to(DEVICE), ytr[idx].float().to(DEVICE)
            loss = F.binary_cross_entropy_with_logits(model(xb).squeeze(1), yb, pos_weight=pos_weight)
            opt.zero_grad()
            loss.backward()
            opt.step()
            sched.step()
            total += loss.item() * len(idx)
            bar.set_postfix(loss=f"{loss.item():.3f}")
        auc = roc_auc_score(yva.numpy(), predict(model, xva))
        if auc > best:
            best = auc
            torch.save({"cfg": cfg, "state": model[1].state_dict()}, runs / "best.pt")
        print(f"ep {ep + 1:2d}  loss {total / len(xtr):.4f}  val_auc {auc:.4f}  {time.time() - t0:.0f}s"
              + ("  *best" if auc == best else ""))

    # Evaluate best checkpoint. Threshold picked on val (Youden's J), then frozen for test/external.
    model[1].load_state_dict(torch.load(runs / "best.pt", map_location="cpu")["state"])
    pva = predict(model, xva)
    fpr, tpr, thr = roc_curve(yva.numpy(), pva)
    t = float(thr[np.argmax(tpr - fpr)])
    out = {"threshold": round(t, 4), "val": metrics(yva.numpy(), pva, t)}
    for name in ["test", "external"]:
        x, y, paths, src = load_split(name, args.clahe)
        p = predict(model, x)
        out[name] = metrics(y.numpy(), p, t)
        np.savez(runs / f"pred_{name}.npz", p=p, y=y.numpy(), path=paths, source=src)
    (runs / "metrics.json").write_text(json.dumps(out, indent=2))
    print(json.dumps(out, indent=2))


if __name__ == "__main__":
    main()


In [ ]:
%%writefile gradcam.py
"""Grad-CAM heatmaps for runs/<name>/best.pt, plus pointing-game score against TBX11K radiologist boxes."""
import argparse
import json
from pathlib import Path

import matplotlib

matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import torch
from matplotlib.patches import Rectangle
from pytorch_grad_cam import GradCAM
from pytorch_grad_cam.utils.model_targets import ClassifierOutputTarget

from train import DEVICE, build_model, load_split, to_input

SCALE = 256 / 512  # boxes are in 512px coords


def target_layer(net):
    if hasattr(net, "layer4"):  # resnet
        return net.layer4[-1]
    if hasattr(net, "conv_head"):  # efficientnet
        return net.conv_head
    return net.features.denseblock4  # xrv densenet (norm5 output is relu'd in place, so hook the block before)


def cams_for(model, x, bs=32):
    layer = target_layer(model[1])
    with GradCAM(model=model, target_layers=[layer]) as cam:
        return np.concatenate([cam(input_tensor=to_input(x[i:i + bs]).to(DEVICE),
                                   targets=[ClassifierOutputTarget(0)] * len(x[i:i + bs]))
                               for i in range(0, len(x), bs)])


def box_mask(bxs):
    m = np.zeros((256, 256), bool)
    for bx, by, bw, bh in np.array(bxs) * SCALE:
        m[int(by):int(np.ceil(by + bh)), int(bx):int(np.ceil(bx + bw))] = True
    return m


def grid(x, cams, titles, boxes, path, cols=4):
    rows = int(np.ceil(len(x) / cols))
    fig, axes = plt.subplots(rows, cols, figsize=(cols * 3, rows * 3.2))
    for ax in np.ravel(axes):
        ax.axis("off")
    for ax, img, c, t, bxs in zip(np.ravel(axes), x, cams, titles, boxes):
        ax.imshow(img, cmap="gray")
        ax.imshow(c, cmap="jet", alpha=0.4)
        for bx, by, bw, bh in np.array(bxs) * SCALE:
            ax.add_patch(Rectangle((bx, by), bw, bh, fill=False, ec="lime", lw=1.5))
        ax.set_title(t, fontsize=8)
    fig.tight_layout()
    fig.savefig(path, dpi=120)
    plt.close(fig)
    print("saved", path)


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--name", default="baseline")
    runs = Path("runs") / ap.parse_args().name
    ck = torch.load(runs / "best.pt", map_location="cpu")
    cfg = ck.get("cfg", {"model": ck.get("model"), "clahe": False})  # first baseline run saved no cfg
    model = build_model(cfg["model"], pretrained=False).to(DEVICE)
    model[1].load_state_dict(ck["state"])
    model.eval()
    thr = json.loads((runs / "metrics.json").read_text())["threshold"]
    boxes = json.loads(open("data/boxes.json").read())

    # 1) Pointing game on test TB images that have radiologist boxes
    x, y, paths, _ = load_split("test", cfg["clahe"])
    p = np.load(runs / "pred_test.npz")["p"]
    tb = [i for i in range(len(y)) if paths[i] in boxes]
    cams = cams_for(model, x[tb])
    masks = [box_mask(boxes[paths[i]]) for i in tb]
    hits = [m.flat[c.argmax()] for c, m in zip(cams, masks)]
    energy = [c[m].sum() / c.sum() for c, m in zip(cams, masks)]
    chance = [m.mean() for m in masks]  # hit rate of a random point = box area fraction
    score = {"n": len(tb), "pointing_game": round(float(np.mean(hits)), 3),
             "random_point_baseline": round(float(np.mean(chance)), 3),
             "cam_energy_in_box": round(float(np.mean(energy)), 3)}
    print(json.dumps(score, indent=2))
    (runs / "pointing_game.json").write_text(json.dumps(score, indent=2))
    show = tb[:12]
    grid(x[show], cams[:12], [f"{paths[i].split('/')[-1]} p={p[i]:.2f} {'HIT' if h else 'miss'}"
                              for i, h in zip(show, hits)],
         [boxes[paths[i]] for i in show], runs / "gradcam_tb.png")

    # 2) Mistakes: missed TB (false negatives) and false alarms (false positives)
    fn = [i for i in np.argsort(p) if y[i] == 1 and p[i] < thr][:4]
    fp = [i for i in np.argsort(-p) if y[i] == 0 and p[i] >= thr][:4]
    idx = fn + fp
    if idx:
        grid(x[idx], cams_for(model, x[idx]),
             [f"{'FN' if y[i] else 'FP'} {paths[i].split('/')[-1]} p={p[i]:.2f}" for i in idx],
             [boxes.get(paths[i], []) for i in idx], runs / "gradcam_errors.png")

    # 3) External hospitals: does attention stay on the lungs on unseen scanners?
    x, y, paths, src = load_split("external", cfg["clahe"])
    pe = np.load(runs / "pred_external.npz")["p"]
    idx = [i for s in np.unique(src) for i in np.flatnonzero(src == s)[np.argsort(-pe[src == s])][:4]]
    grid(x[idx], cams_for(model, x[idx]),
         [f"{src[i]} y={y[i]} p={pe[i]:.2f}" for i in idx], [[]] * len(idx), runs / "gradcam_external.png")


if __name__ == "__main__":
    main()


In [ ]:
%%writefile compare.py
"""Side-by-side table of every experiment in runs/: internal test vs each external hospital vs Grad-CAM localization."""
import json
from pathlib import Path

import numpy as np
from sklearn.metrics import roc_auc_score


def site(path):
    return "Montgomery" if "MCUCXR" in path else "Shenzhen" if "CHNCXR" in path else "DA/DB"


cols = ["test", "external", "DA/DB", "Montgomery", "Shenzhen", "pointing", "energy"]
print(f"{'run':16s}" + "".join(f"{c:>11s}" for c in cols))
for run in sorted(Path("runs").glob("*/pred_external.npz")):
    d = run.parent
    t, e = np.load(d / "pred_test.npz"), np.load(d / "pred_external.npz")
    sites = np.array([site(p) for p in e["path"]])
    row = [roc_auc_score(t["y"], t["p"]), roc_auc_score(e["y"], e["p"])]
    row += [roc_auc_score(e["y"][sites == s], e["p"][sites == s]) for s in cols[2:5]]
    pg = json.loads((d / "pointing_game.json").read_text()) if (d / "pointing_game.json").exists() else {}
    row += [pg.get("pointing_game", np.nan), pg.get("cam_energy_in_box", np.nan)]
    print(f"{d.name:16s}" + "".join(f"{v:11.3f}" for v in row))
print("AUC columns: test..Shenzhen | Grad-CAM vs radiologist boxes: pointing (random ~0.106), energy-in-box")


def boot_delta(y, pa, pb, n=1000, seed=0):
    """Paired, label-stratified bootstrap 95% CI of AUC(pa) - AUC(pb) on the same images."""
    rng = np.random.default_rng(seed)
    pos, neg = np.flatnonzero(y == 1), np.flatnonzero(y == 0)
    d = []
    for _ in range(n):
        i = np.concatenate([rng.choice(pos, len(pos)), rng.choice(neg, len(neg))])
        d.append(roc_auc_score(y[i], pa[i]) - roc_auc_score(y[i], pb[i]))
    return np.percentile(d, [2.5, 97.5])


# Is a run's external gain real or noise? CI excluding 0 = difference unlikely to be chance (single seed though).
if not Path("runs/baseline/pred_external.npz").exists():  # e.g. on Colab, where only the remote runs live
    raise SystemExit
base = np.load("runs/baseline/pred_external.npz")
sites = np.array([site(p) for p in base["path"]])
groups = {"external": np.ones(len(sites), bool)} | {s: sites == s for s in cols[2:5]}
print(f"\n{'Δ AUC vs baseline':18s}" + "".join(f"{g:>24s}" for g in groups))
for run in sorted(Path("runs").glob("*/pred_external.npz")):
    if run.parent.name == "baseline":
        continue
    e = np.load(run)
    assert (e["path"] == base["path"]).all(), f"{run}: external images differ from baseline"
    cells = []
    for m in groups.values():
        y, pa, pb = e["y"][m], e["p"][m], base["p"][m]
        lo, hi = boot_delta(y, pa, pb)
        star = "*" if lo > 0 or hi < 0 else " "
        cells.append(f"{roc_auc_score(y, pa) - roc_auc_score(y, pb):+.3f} [{lo:+.3f},{hi:+.3f}]{star}")
    print(f"{run.parent.name:18s}" + "".join(f"{c:>24s}" for c in cells))
print("[95% bootstrap CI]  * = CI excludes 0")


In [ ]:
!python prepare.py  # downloads TBX11K (~3GB) and builds the same deterministic splits as the Mac

In [ ]:
import hashlib, numpy as np
local = {'train': '74ce0154248abd53a0fbbcb5a6f228ae', 'val': '54963658a393cea1c83b716146313ee9', 'test': 'cdc649d901f62c7df10c51f11a119327', 'external': 'acbee31572e5e3c7b9417ef312864707'}
colab = {s: hashlib.md5('|'.join(np.load(f'data/{s}.npz')['path']).encode()).hexdigest() for s in local}
assert colab == local, f'split mismatch vs Mac: {colab}'
print('splits identical to Mac')

In [ ]:
!python -u train.py --name AB_xrv_clahe_aug --model xrv-densenet --clahe --strong-aug
!python gradcam.py --name AB_xrv_clahe_aug

In [ ]:
!python compare.py
!zip -qr runs_AB_xrv_clahe_aug.zip runs/AB_xrv_clahe_aug
from google.colab import files
files.download('runs_AB_xrv_clahe_aug.zip')